# 03 — Modelo predictivo de deserción estudiantil

Entrena y valida el modelo de deserción sobre `gold/fact_estudiante_semestre`
(grano estudiante × semestre, target `desertion_t1` censurado en el último semestre).

**Protocolo (idéntico al job `ml_train_desertion.py` — mantener en sincronía):**
1. Holdout PRIMERO: `GroupShuffleSplit` 80/20 agrupado por `id`.
2. `StratifiedGroupKFold` k=5 **dentro** del 80% (OOF para tuning).
3. Threshold: máximo recall sujeto a precisión ≥ 0.80 sobre OOF (fallback: máximo F1, documentado).
4. Holdout evaluado UNA sola vez con el threshold congelado.
5. Validación temporal: entrenar semestres tempranos → evaluar el último etiquetado.
6. Baseline `LogisticRegression`; comparación `HistGradientBoosting` vs `XGBoost`.
7. SHAP para las figuras de explicabilidad de la tesis (solo notebook, no en el job).

**Entorno espejo de Glue Python Shell** (para reproducibilidad del artefacto):
```bash
python3.9 -m venv .venv-ml && source .venv-ml/bin/activate
pip install "scikit-learn==1.0.2" "pandas==1.4.2" "numpy==1.22.3" awswrangler joblib shap xgboost matplotlib
```
**Leakage — columnas prohibidas como features:** `total_periodos_matriculados`,
`tasa_permanencia`, `primer_periodo`, `ultimo_periodo`, `next_semestre_orden`,
`semestre_orden`, `anio` (proxies del calendario/censura) y `municipio_residencia`
(alta cardinalidad; el contexto municipal ya entra vía las métricas joineadas).

In [ ]:
import json
import numpy as np
import pandas as pd
import awswrangler as wr
import matplotlib.pyplot as plt
import sklearn

GOLD_BUCKET = "data-lake-academico-gold-462035739083"
RANDOM_STATE = 42
MIN_PRECISION = 0.80

print("sklearn:", sklearn.__version__, "| pandas:", pd.__version__)
df = wr.s3.read_parquet(path=f"s3://{GOLD_BUCKET}/fact_estudiante_semestre/")
print(df.shape)
df.head(3)

## 1. Validación post-fixes (curva de deserción por semestre)
Si la tasa de los semestres tardíos se dispara frente a los tempranos, hay **censura residual** y NO se debe entrenar todavía.

In [ ]:
df["desertion_t1"] = pd.to_numeric(df["desertion_t1"], errors="coerce")
resumen = df.groupby("semestre_orden").agg(
    n=("id", "count"),
    estudiantes=("id", "nunique"),
    tasa_desercion=("desertion_t1", "mean"),
    censurados=("desertion_t1", lambda s: s.isna().mean()),
)
display(resumen)

cobertura = df["codigo_divipola"].notna().mean()
print(f"Cobertura DIVIPOLA/socioeconómica: {cobertura:.1%}  (criterio de aceptación: >= 90%)")
print(f"Prevalencia global (etiquetados): {df['desertion_t1'].mean():.1%}  (esperado: 12-16%)")

ax = resumen["tasa_desercion"].plot(kind="bar", color="#004B8D", figsize=(7, 3.5),
                                     title="Tasa de deserción por semestre (validación de censura)")
ax.set_ylabel("tasa")
plt.tight_layout(); plt.show()

## 2. Features (idéntico al job)

In [ ]:
NUMERIC_FEATURES = [
    "edad", "estrato_social", "n_periodos_en_semestre",
    "semestres_cursados_acum", "gap_desde_semestre_anterior", "es_primer_semestre",
    "tasa_informalidad_mpio", "tasa_hacinamiento_mpio", "promedio_ipm_mpio",
    "log_poblacion_sisben_mpio", "log_total_accesos_res", "nivel_conectividad",
]
CATEGORICAL_FEATURES = [
    "sexo", "zona_de_residencia", "escuela", "programa", "zona", "centro",
    "departamento_residencia", "cohorte_tipo",
]
TARGET = "desertion_t1"
UNKNOWN_CATEGORY, OTHER_CATEGORY, MAX_CATEGORIES = "DESCONOCIDO", "OTRO", 200


def build_features(data):
    out = data.copy()
    for col in ["edad", "estrato_social", "nivel_conectividad", "n_periodos_en_semestre",
                "semestres_cursados_acum", "gap_desde_semestre_anterior", "es_primer_semestre"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["log_poblacion_sisben_mpio"] = np.log1p(pd.to_numeric(out["poblacion_sisben_mpio"], errors="coerce"))
    out["log_total_accesos_res"] = np.log1p(pd.to_numeric(out["total_accesos_res"], errors="coerce"))
    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].fillna(UNKNOWN_CATEGORY).astype(str).str.strip().str.upper()
    return out


def cap_categories(train_data):
    return {c: sorted(set(train_data[c].value_counts().index[:MAX_CATEGORIES]) | {OTHER_CATEGORY, UNKNOWN_CATEGORY})
            for c in CATEGORICAL_FEATURES}


def apply_category_map(data, cats):
    out = data.copy()
    for c, allowed in cats.items():
        out[c] = out[c].where(out[c].isin(allowed), OTHER_CATEGORY)
    return out


data = build_features(df)
labeled = data[data[TARGET].notna()].copy()
labeled[TARGET] = labeled[TARGET].astype(int)
print(len(labeled), "filas etiquetadas |", labeled['id'].nunique(), "estudiantes")

## 3. Split (holdout primero), CV agrupado y threshold

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight


def make_model(cats):
    cat_lists = [cats[c] for c in CATEGORICAL_FEATURES]
    encoder = ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OrdinalEncoder(categories=cat_lists, handle_unknown="use_encoded_value",
                               unknown_value=max(len(c) for c in cat_lists)), CATEGORICAL_FEATURES),
    ])
    clf = HistGradientBoostingClassifier(
        categorical_features=[False] * len(NUMERIC_FEATURES) + [True] * len(CATEGORICAL_FEATURES),
        random_state=RANDOM_STATE)
    return Pipeline([("encoder", encoder), ("clf", clf)])


def fit_weighted(pipe, X, y):
    pipe.fit(X, y, clf__sample_weight=compute_sample_weight("balanced", y))
    return pipe


def tune_threshold(y_true, y_score):
    p, r, t = precision_recall_curve(y_true, y_score)
    p, r = p[:-1], r[:-1]
    feasible = p >= MIN_PRECISION
    if feasible.any():
        i = np.argmax(np.where(feasible, r, -1.0))
        return float(t[i]), f"max recall s.a. precision>={MIN_PRECISION}"
    f1 = 2 * p * r / np.clip(p + r, 1e-9, None)
    return float(t[int(np.argmax(f1))]), "max F1 (precision objetivo inalcanzable en OOF)"


def evaluate(y_true, y_score, thr):
    y_pred = (y_score >= thr).astype(int)
    return {"precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_score),
            "pr_auc": average_precision_score(y_true, y_score),
            "cm": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()}


tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
                      .split(labeled, labeled[TARGET], groups=labeled["id"]))
train_df, test_df = labeled.iloc[tr_idx], labeled.iloc[te_idx]
cats = cap_categories(train_df)
train_df, test_df = apply_category_map(train_df, cats), apply_category_map(test_df, cats)
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X_train, y_train = train_df[FEATURES], train_df[TARGET].values
X_test, y_test = test_df[FEATURES], test_df[TARGET].values

oof = np.full(len(train_df), np.nan)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for k, (fi, vi) in enumerate(cv.split(X_train, y_train, groups=train_df["id"])):
    mk = fit_weighted(make_model(cats), X_train.iloc[fi], y_train[fi])
    oof[vi] = mk.predict_proba(X_train.iloc[vi])[:, 1]
    print(f"fold {k}: prevalencia={y_train[vi].mean():.3f} | pr_auc={average_precision_score(y_train[vi], oof[vi]):.3f}")

THRESHOLD, criterio = tune_threshold(y_train, oof)
print(f"\nThreshold={THRESHOLD:.4f} ({criterio})")
print("OOF:", evaluate(y_train, oof, THRESHOLD))

## 4. Holdout (una sola vez), baseline y XGBoost comparativo

In [ ]:
model = fit_weighted(make_model(cats), X_train, y_train)
test_scores = model.predict_proba(X_test)[:, 1]
print("HistGB holdout:", evaluate(y_test, test_scores, THRESHOLD))

baseline = Pipeline([
    ("encoder", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))])
baseline.fit(X_train, y_train)
print("Baseline LogReg holdout:", evaluate(y_test, baseline.predict_proba(X_test)[:, 1], THRESHOLD))

# XGBoost comparativo (la tesis lo menciona; el job productivo usa HistGB por no requerir pip)
try:
    from xgboost import XGBClassifier
    xgb = Pipeline([
        ("encoder", make_model(cats).named_steps["encoder"]),
        ("clf", XGBClassifier(n_estimators=400, learning_rate=0.08, max_depth=6,
                               subsample=0.9, colsample_bytree=0.9, eval_metric="aucpr",
                               scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
                               random_state=RANDOM_STATE, n_jobs=-1))])
    xgb.fit(X_train, y_train)
    print("XGBoost holdout:", evaluate(y_test, xgb.predict_proba(X_test)[:, 1], THRESHOLD))
except ImportError:
    print("xgboost no instalado — comparación omitida")

## 5. Validación temporal (métrica honesta de generalización a futuro)

In [ ]:
orders = sorted(labeled["semestre_orden"].unique())
tr_t = apply_category_map(labeled[labeled["semestre_orden"].isin(orders[:-1])], cats)
te_t = apply_category_map(labeled[labeled["semestre_orden"] == orders[-1]], cats)
model_t = fit_weighted(make_model(cats), tr_t[FEATURES], tr_t[TARGET].values)
temporal = evaluate(te_t[TARGET].values, model_t.predict_proba(te_t[FEATURES])[:, 1], THRESHOLD)
print(f"Temporal (train semestres {orders[:-1]} -> test semestre {orders[-1]}):", temporal)
print("\nSi la métrica temporal cae mucho frente al CV agrupado, discutirlo",
      "honestamente en la tesis (drift de calendario / censura residual).")

## 6. Explicabilidad SHAP (figuras para la tesis)

In [ ]:
import shap

encoder = model.named_steps["encoder"]
clf = model.named_steps["clf"]
X_test_enc = pd.DataFrame(encoder.transform(X_test), columns=FEATURES)
sample = X_test_enc.sample(min(5000, len(X_test_enc)), random_state=RANDOM_STATE)

explainer = shap.Explainer(clf, sample, feature_names=FEATURES)
shap_values = explainer(sample)

shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("Impacto de las variables en el riesgo de deserción (SHAP)")
plt.tight_layout(); plt.savefig("shap_beeswarm_desercion.png", dpi=200); plt.show()

shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout(); plt.savefig("shap_importancia_desercion.png", dpi=200); plt.show()

## 7. Sesgo algorítmico y conclusión
- Comparar tasas de falsos negativos/positivos por `estrato_social`, `sexo` y `zona_de_residencia`
  (equidad: el modelo no debe concentrar errores en los grupos vulnerables).
- Si el holdout no alcanza F1 ≥ 0.80 y Precisión ≥ 0.80: registrar el punto de operación
  alcanzado, la curva PR completa y el lift vs baseline, y **acordar con el director** la
  enmienda de métrica ANTES de la sustentación (right-censoring + ausencia de variables
  académicas acotan el techo). Prohibido reincluir leakage o tunear sobre el holdout.

In [ ]:
test_eval = test_df.copy()
test_eval["score"] = test_scores
test_eval["pred"] = (test_eval["score"] >= THRESHOLD).astype(int)
for dim in ["estrato_social", "sexo", "zona_de_residencia"]:
    g = test_eval.groupby(dim).apply(
        lambda d: pd.Series({
            "n": len(d),
            "prevalencia": d[TARGET].mean(),
            "recall": recall_score(d[TARGET], d["pred"], zero_division=0),
            "precision": precision_score(d[TARGET], d["pred"], zero_division=0),
        }))
    print(f"\n=== Equidad por {dim} ===")
    display(g[g["n"] >= 100].round(3))